## Get nucleotide frequencies of Tuberculosis genome
Mycobacterium tuberculosis H37Rv

In [3]:
! wget "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000962.3&rettype=fasta&retmode=text" -O data/NC_000962.3.fasta

--2025-06-27 12:28:01--  https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000962.3&rettype=fasta&retmode=text
Resolving eutils.ncbi.nlm.nih.gov (eutils.ncbi.nlm.nih.gov)... 2607:f220:41e:4290::110, 130.14.29.110
Connecting to eutils.ncbi.nlm.nih.gov (eutils.ncbi.nlm.nih.gov)|2607:f220:41e:4290::110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘data/NC_000962.3.fasta’

data/NC_000962.3.fa     [    <=>             ]   4,27M  5,27MB/s    in 0,8s    

2025-06-27 12:28:03 (5,27 MB/s) - ‘data/NC_000962.3.fasta’ saved [4474618]



In [58]:
from collections import Counter
import numpy as np
import pandas as pd
from Bio import SeqIO
import sys

In [12]:
def calculate_nucleotide_frequencies(fasta_path):
    """
    Calculates nucleotide fequencies of a genome.
    Would also work for a fasta file with more than one sequence, even though I'm using it for a whole genome fasta file.
    """
    total_counts = Counter()

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq).upper()
        total_counts.update(seq)

    total_nucleotides = sum(total_counts[nuc] for nuc in "ACGT")
    frequencies = {nuc: total_counts[nuc] / total_nucleotides for nuc in "ACGT"}

    print("Nucleotide Frequencies:")
    for nuc in "ACGT":
        print(f"{nuc}: {frequencies[nuc]:.4f}")
    return frequencies

freq = calculate_nucleotide_frequencies("data/NC_000962.3.fasta")

Nucleotide Frequencies:
A: 0.1719
C: 0.3287
G: 0.3275
T: 0.1719


In [17]:
def create_hky_q_matrix(frequencies, kappa):
    """
    Creates the scaled HKY Q matrix such that the expected substitution rate is 1.
    Rows and columns are ordered as A, C, G, T.
    """
    nucs = ['A', 'C', 'G', 'T']
    pi = [frequencies[nuc] for nuc in nucs]

    # Initialize Q matrix
    Q = np.zeros((4, 4))

    # Define transitions and transversions
    transitions = {('A', 'G'), ('G', 'A'), ('C', 'T'), ('T', 'C')}

    for i, ni in enumerate(nucs):
        for j, nj in enumerate(nucs):
            if i == j:
                continue
            if (ni, nj) in transitions:
                Q[i, j] = kappa * frequencies[nj]
            else:
                Q[i, j] = frequencies[nj]

    # Set diagonal entries so rows sum to zero
    for i in range(4):
        Q[i, i] = -np.sum(Q[i, :])

    # Scale so that the expected substitution rate is 1
    total_rate = -sum(pi[i] * Q[i, i] for i in range(4))
    Q /= total_rate

    return Q, pd.DataFrame(Q, index=nucs, columns=nucs)

In [23]:
Q, Q_df = create_hky_q_matrix(freq, kappa = 2.07)
Q

array([[-1.21861714,  0.3398901 ,  0.70096006,  0.17776698],
       [ 0.17781012, -0.88441582,  0.33862805,  0.36797766],
       [ 0.36806694,  0.3398901 , -0.88572402,  0.17776698],
       [ 0.17781012,  0.7035725 ,  0.33862805, -1.22001066]])

In [24]:
# scale Q by substitution rate
mu = 4.6e-8
Q = mu * Q
Q

array([[-5.60563883e-08,  1.56349444e-08,  3.22441626e-08,
         8.17728129e-09],
       [ 8.17926531e-09, -4.06831277e-08,  1.55768902e-08,
         1.69269723e-08],
       [ 1.69310792e-08,  1.56349444e-08, -4.07433049e-08,
         8.17728129e-09],
       [ 8.17926531e-09,  3.23643349e-08,  1.55768902e-08,
        -5.61204904e-08]])

Seq-Gen interprets branch lengths as units of expected substitutions per site. If a branch length is 0.1, Seq-Gen treats it as 0.1 expected substitutions per site.

Suppose: A branch length in your tree is 1.2 years. The substitution rate is μ=4.6×10−8 substitutions/site/year. To tell Seq-Gen: “this branch evolves for 1.2 years at this rate,” you compute: Substitutions/site on this branch=μ⋅time=4.6×10−8⋅1.2=5.52×10−8
→ This is the value that Seq-Gen needs as the branch length

In [6]:
2000 * np.exp(-200*0.02)

np.float64(36.631277777468355)

In [71]:
from Bio import Nexus, Phylo
from io import StringIO

In [51]:
treepath = "/Users/mariebecker/Documents/Uni/ETH/RotationStadler/BESP_paper-analyses/results/pop_size_simulations/linear_constant/uniform/uniform_0.tree"
tree = Phylo.read(treepath, "newick")
#Phylo.write(tree, "converted.nex", "nexus")

In [52]:
tree.get_terminals()

[Clade(branch_length=85.51169686, name='t123'),
 Clade(branch_length=85.20077156, name='t124'),
 Clade(branch_length=29.40221713, name='t125'),
 Clade(branch_length=29.20843118, name='t126'),
 Clade(branch_length=244.644568, name='t127'),
 Clade(branch_length=26.18635088, name='t128'),
 Clade(branch_length=26.14262086, name='t129'),
 Clade(branch_length=21.42206384, name='t130'),
 Clade(branch_length=19.96622763, name='t131'),
 Clade(branch_length=36.97792539, name='t132'),
 Clade(branch_length=24.31275061, name='t133'),
 Clade(branch_length=15.27725582, name='t134'),
 Clade(branch_length=14.6616526, name='t135'),
 Clade(branch_length=21.20150853, name='t136'),
 Clade(branch_length=43.9680417, name='t137'),
 Clade(branch_length=12.81287944, name='t138'),
 Clade(branch_length=11.90913365, name='t139'),
 Clade(branch_length=6.631340953, name='t140'),
 Clade(branch_length=6.186669348, name='t141'),
 Clade(branch_length=282.3984372, name='t142'),
 Clade(branch_length=24.8067227, name='t143

In [ ]:
            # Read alignment and format as BEAST-compatible XML block
            alignment = AlignIO.read(aln_path, "fasta")
            output_align = io.StringIO()
            for record in alignment:
                #output_align.write(f'\n\t\t\t\t\t<sequence spec="Sequence" taxon="{record.id}" value="{str(record.seq)}" />')
                output_align.write(f'\n\t<sequence><taxon idref="{record.id}"/>{str(record.seq)}</sequence>')
            # Set parameters not in the config file
            pars["taxa"] = output_align.getvalue()

In [ ]:
def pars_dates(tree, pars):
    """
    Add dates to the tree in the BEAST XML parameters.
    """
    output_dates = StringIO()

    tree_ = Phylo.read(StringIO(tree), "newick")
    leaves = tree_.get_terminals()

    # Compute distances from root to each leaf (sampling height)
    heights = np.array([tree_.distance(leaf) for leaf in leaves])
    treetimes = np.max(heights) - heights
    leaf_names = [leaf.name for leaf in leaves]

    for i, leaf_name in enumerate(leaf_names):
        output_dates.write(f'\n\t<taxon id = "{leaf_name}"> <date value="{treetimes[i]}" direction="backwards" units="years"/> </taxon>')

    pars["dates"] = output_dates.getvalue()

In [72]:
tree_line = "(((((((t123:85.51169686,t124:85.20077156):7.343365349,(t125:29.40221713,t126:29.20843118):63.11290296):289.956407,(t127:244.644568,(((t128:26.18635088,t129:26.14262086):13.84757867,((t130:21.42206384,t131:19.96622763):17.34847746,t132:36.97792539):1.058985511):9.845577978,((t133:24.31275061,((t134:15.27725582,t135:14.6616526):6.5619838,t136:21.20150853):2.267177142):21.00649058,t137:43.9680417):2.524411378):194.6177663):137.1583196):403.4837819,((t138:12.81287944,t139:11.90913365):43.38167875,(t140:6.631340953,t141:6.186669348):48.3076185):724.9651857):828.4140678,(t142:282.3984372,((t143:24.8067227,t144:24.01293535):181.7886839,(t145:1.665177037,t146:1.233016773):204.0694754):75.77391343):1324.583421):1784.875949,(((((t147:5.075415489,t148:4.331392641):5.0400927,t149:9.287065444):69.71574544,(t150:25.77983416,(t151:23.14353761,t152:23.01059492):2.625521291):51.67712957):38.4252127,(t153:39.32858169,t154:38.86890371):76.08494234):450.3432882,((((((t155:6.501519402,(t156:5.112483802,t157:4.798736082):1.178995211):11.67523512,t158:16.76180654):4.574980565,(t159:11.51923451,t160:10.72834069):9.105124658):31.0904289,t161:50.77555151):2.85343334,(t162:18.15745443,t163:17.95611744):34.94200278):283.8208794,(((((t164:18.00394497,(t165:6.370289088,t166:5.935099884):11.47095318):99.62947934,((t167:20.07805183,t168:19.66252073):31.89128786,(t169:50.2751031,t170:49.96946908):1.127506835):64.8118296):105.4726258,((((t171:20.7640469,t172:20.55289959):26.83457463,(t173:3.132601653,t174:3.081646737):43.68631935):37.09070349,((t175:10.60626825,(t176:8.38900566,t177:8.221200168):2.215046601):66.37127935,t178:75.9061895):5.801951552):84.9812876,((((t179:12.93423768,t180:11.948838):13.82790165,(t181:19.73975543,t182:18.66957601):5.456336538):34.62004259,(t183:50.89858956,(t184:19.70217632,t185:19.66577465):30.59006162):7.502177309):37.79863801,((t186:10.86301823,t187:10.85438142):9.659181448,t188:20.13616126):74.58712515):67.36717053):51.42396909):53.64542889,(t189:246.7057591,((((t190:35.5785022,t191:35.23160107):13.43229019,(t192:24.6913429,t193:23.47929574):23.21268286):9.600603416,((((t1:97.06484683,(((t2:23.71789582,t3:23.66248825):55.6411441,(t4:74.61738058,(t5:72.03945324,((t6:24.52738835,(t7:8.200710303,t8:8.164346021):16.16725409):42.08599551,(t9:55.88317411,t10:55.20569545):9.905839193):5.0109684):2.506353574):2.887407175):7.039082311,(t11:81.32493379,(((t12:9.876600217,t13:9.0138726):5.733114758,t14:14.58361624):39.51343857,t15:53.90269216):26.00434397):0.8948687282):10.62741294):23.77375871,t16:115.1695439):6.809147444,(((t17:46.60402976,t18:45.41464108):13.23009948,t19:58.53418386):20.24717966,(t20:33.30585927,t21:33.17455168):44.51403098):41.49041021):4.840497746,((((t22:9.777849086,t23:9.220883541):4.966160145,t24:13.82950755):25.56178769,(t25:37.21088317,t26:36.29725749):1.696076883):58.02135063,t27:95.65959859):25.63917925):22.71679972):105.9811786,t28:249.7336229):81.85274293):19.98247445):14.96787122,((((t29:51.70433703,(t30:5.006060849,t31:4.936402344):46.6527271):23.98630916,t32:75.26662012):146.3987774,((((t33:5.813309351,t34:5.13642394):24.36889788,t35:29.41664922):44.99042741,(t36:46.83971593,(t37:36.61307198,t38:36.39586597):9.784806596):27.15995883):50.95413607,(((t39:42.13160476,((t40:15.75226698,t41:15.61261264):4.757455761,(t42:1.289628205,t43:1.09493075):18.35219996):21.40773715):78.71378972,(t44:61.03365566,t45:59.68617227):58.01999681):3.191992108,(t46:28.79180285,t47:27.54833082):91.03818126):0.09149099181):95.43945306):132.9479019,(((((((((t48:13.78290449,t49:12.96215908):25.22755982,(t50:3.939738813,t51:2.821911699):33.87850762):1.586335351,t52:38.28066226):18.08921491,t53:55.14172672):9.033134053,(t54:46.51816875,t55:44.45062912):17.47675668):2.602648221,((t56:38.20072439,((t57:20.78062896,t58:20.76054516):10.99659374,t59:31.36248841):5.722765268):19.53237375,(t60:28.64030657,t61:28.58802601):25.44568922):6.368856928):1.866470108,((t62:31.0128803,t63:30.57939316):19.86012033,t64:48.48085609):10.25804289):200.4307214,(t65:23.99137963,t66:22.94765373):235.1381662):2.885629019,(((t67:56.9869927,t68:54.1205658):44.41857206,((t69:5.414493369,(t70:4.784105514,t71:4.607163957):0.3073624622):51.42416671,(t72:13.05675177,((t73:0.464911443,t74:0.02503098068):7.39178421,t75:3.184212648):4.491543783):43.24784729):41.64969963):67.37704604,(((t76:43.44011173,(t77:27.26108832,t78:26.83983081):15.01313293):1.955716795,(t79:38.60443704,t80:38.34696568):3.653345889):88.08571429,(t81:115.0666402,t82:114.5463357):10.33256969):26.27589871):91.11690811):71.31895093):11.16830144):44.81597559):224.6695928):2821.700378):354.9380153,((((t83:75.38087545,t84:74.12948115):141.9932271,(((t85:46.56071837,t86:46.28274245):16.65469849,t87:62.2711137):24.34510691,((t88:43.61453378,t89:43.61004904):41.81020001,((t90:38.02855111,t91:37.63886174):25.6182063,t92:63.21307195):21.44263371):1.001441999):128.294883):180.1642534,((((t93:11.49891061,t94:11.3677606):61.49801816,((t95:11.93107304,(t96:4.626101633,t97:4.615104995):6.647926307):42.69934083,t98:53.83920579):17.92581868):67.56357924,(((t99:24.25271397,t100:24.08429857):28.3772211,(((t101:12.83102102,t102:12.5090022):22.17070991,(t103:11.8367177,t104:11.05793691):22.52271151):0.5620867758,t105:33.76110887):15.17674998):1.054430019,((t106:11.89124138,t107:11.74834602):8.169461656,t108:18.95484656):29.89984851):85.12034301):47.66548495,(t109:13.66845066,t110:13.42775698):167.9487691):205.8565257):232.0299868,(((t111:8.937486367,t112:7.830085059):75.23544991,t113:82.52951569):455.1592802,((((t114:4.817223241,t115:4.299202952):38.79873362,((t116:14.53520568,t117:14.22710676):2.514260427,t118:16.48381994):25.79467207):3.144691536,(t119:24.49914135,t120:24.42420434):20.80350828):49.25106912,(t121:42.89815647,t122:41.3437511):51.36787791):441.3089666):79.83073337):3141.157481)"
tree = Phylo.read(StringIO(tree_line), "newick")

leaves = tree.get_terminals()

# Compute distances from root to each leaf (sampling height)
heights = np.array([tree.distance(leaf) for leaf in leaves])
treetimes = max(heights) - heights
leaf_names = [leaf.name for leaf in leaves]

leaf_names = [leaf.name for leaf in leaves]
sampling_df = pd.DataFrame({
    'name': leaf_names,
    'sampling_time': treetimes
})

sampling_df.to_csv("data/sampling_times.csv", index=False, sep='\t')

In [ ]:
# Map names to sampling times
sampling_times = {leaf.name: time for leaf, time in zip(leaves, treetimes)}

In [ ]:

df

,name,sampling_time
0,t123,69.589996
1,t124,69.900921
2,t125,69.929938
3,t126,70.123724
4,t127,70.598578
...,...,...
188,t118,67.141846
189,t119,67.262380
190,t120,67.337317
191,t121,67.550064


In [ ]:
treetimes = max(heights) - heights

sampling_times = {leaf.name: time for leaf, time in zip(leaves, treetimes)}
pd.DataFrame.from_dict(sampling_times, orient='index', columns=['sampling_time']).sort_values('sampling_time').reset_index()

,index,sampling_time
0,t1,0.000000
1,t2,0.039312
2,t3,0.094719
3,t4,1.893564
4,t5,1.965138
...,...,...
188,t189,96.333213
189,t190,96.593655
190,t191,96.940556
191,t192,97.700421


In [38]:

tree_text = "(((((((t123:85.51169686,t124:85.20077156):7.343365349,(t125:29.40221713,t126:29.20843118):63.11290296):289.956407,(t127:244.644568,(((t128:26.18635088,t129:26.14262086):13.84757867,((t130:21.42206384,t131:19.96622763):17.34847746,t132:36.97792539):1.058985511):9.845577978,((t133:24.31275061,((t134:15.27725582,t135:14.6616526):6.5619838,t136:21.20150853):2.267177142):21.00649058,t137:43.9680417):2.524411378):194.6177663):137.1583196):403.4837819,((t138:12.81287944,t139:11.90913365):43.38167875,(t140:6.631340953,t141:6.186669348):48.3076185):724.9651857):828.4140678,(t142:282.3984372,((t143:24.8067227,t144:24.01293535):181.7886839,(t145:1.665177037,t146:1.233016773):204.0694754):75.77391343):1324.583421):1784.875949,(((((t147:5.075415489,t148:4.331392641):5.0400927,t149:9.287065444):69.71574544,(t150:25.77983416,(t151:23.14353761,t152:23.01059492):2.625521291):51.67712957):38.4252127,(t153:39.32858169,t154:38.86890371):76.08494234):450.3432882,((((((t155:6.501519402,(t156:5.112483802,t157:4.798736082):1.178995211):11.67523512,t158:16.76180654):4.574980565,(t159:11.51923451,t160:10.72834069):9.105124658):31.0904289,t161:50.77555151):2.85343334,(t162:18.15745443,t163:17.95611744):34.94200278):283.8208794,(((((t164:18.00394497,(t165:6.370289088,t166:5.935099884):11.47095318):99.62947934,((t167:20.07805183,t168:19.66252073):31.89128786,(t169:50.2751031,t170:49.96946908):1.127506835):64.8118296):105.4726258,((((t171:20.7640469,t172:20.55289959):26.83457463,(t173:3.132601653,t174:3.081646737):43.68631935):37.09070349,((t175:10.60626825,(t176:8.38900566,t177:8.221200168):2.215046601):66.37127935,t178:75.9061895):5.801951552):84.9812876,((((t179:12.93423768,t180:11.948838):13.82790165,(t181:19.73975543,t182:18.66957601):5.456336538):34.62004259,(t183:50.89858956,(t184:19.70217632,t185:19.66577465):30.59006162):7.502177309):37.79863801,((t186:10.86301823,t187:10.85438142):9.659181448,t188:20.13616126):74.58712515):67.36717053):51.42396909):53.64542889,(t189:246.7057591,((((t190:35.5785022,t191:35.23160107):13.43229019,(t192:24.6913429,t193:23.47929574):23.21268286):9.600603416,((((t1:97.06484683,(((t2:23.71789582,t3:23.66248825):55.6411441,(t4:74.61738058,(t5:72.03945324,((t6:24.52738835,(t7:8.200710303,t8:8.164346021):16.16725409):42.08599551,(t9:55.88317411,t10:55.20569545):9.905839193):5.0109684):2.506353574):2.887407175):7.039082311,(t11:81.32493379,(((t12:9.876600217,t13:9.0138726):5.733114758,t14:14.58361624):39.51343857,t15:53.90269216):26.00434397):0.8948687282):10.62741294):23.77375871,t16:115.1695439):6.809147444,(((t17:46.60402976,t18:45.41464108):13.23009948,t19:58.53418386):20.24717966,(t20:33.30585927,t21:33.17455168):44.51403098):41.49041021):4.840497746,((((t22:9.777849086,t23:9.220883541):4.966160145,t24:13.82950755):25.56178769,(t25:37.21088317,t26:36.29725749):1.696076883):58.02135063,t27:95.65959859):25.63917925):22.71679972):105.9811786,t28:249.7336229):81.85274293):19.98247445):14.96787122,((((t29:51.70433703,(t30:5.006060849,t31:4.936402344):46.6527271):23.98630916,t32:75.26662012):146.3987774,((((t33:5.813309351,t34:5.13642394):24.36889788,t35:29.41664922):44.99042741,(t36:46.83971593,(t37:36.61307198,t38:36.39586597):9.784806596):27.15995883):50.95413607,(((t39:42.13160476,((t40:15.75226698,t41:15.61261264):4.757455761,(t42:1.289628205,t43:1.09493075):18.35219996):21.40773715):78.71378972,(t44:61.03365566,t45:59.68617227):58.01999681):3.191992108,(t46:28.79180285,t47:27.54833082):91.03818126):0.09149099181):95.43945306):132.9479019,(((((((((t48:13.78290449,t49:12.96215908):25.22755982,(t50:3.939738813,t51:2.821911699):33.87850762):1.586335351,t52:38.28066226):18.08921491,t53:55.14172672):9.033134053,(t54:46.51816875,t55:44.45062912):17.47675668):2.602648221,((t56:38.20072439,((t57:20.78062896,t58:20.76054516):10.99659374,t59:31.36248841):5.722765268):19.53237375,(t60:28.64030657,t61:28.58802601):25.44568922):6.368856928):1.866470108,((t62:31.0128803,t63:30.57939316):19.86012033,t64:48.48085609):10.25804289):200.4307214,(t65:23.99137963,t66:22.94765373):235.1381662):2.885629019,(((t67:56.9869927,t68:54.1205658):44.41857206,((t69:5.414493369,(t70:4.784105514,t71:4.607163957):0.3073624622):51.42416671,(t72:13.05675177,((t73:0.464911443,t74:0.02503098068):7.39178421,t75:3.184212648):4.491543783):43.24784729):41.64969963):67.37704604,(((t76:43.44011173,(t77:27.26108832,t78:26.83983081):15.01313293):1.955716795,(t79:38.60443704,t80:38.34696568):3.653345889):88.08571429,(t81:115.0666402,t82:114.5463357):10.33256969):26.27589871):91.11690811):71.31895093):11.16830144):44.81597559):224.6695928):2821.700378):354.9380153,((((t83:75.38087545,t84:74.12948115):141.9932271,(((t85:46.56071837,t86:46.28274245):16.65469849,t87:62.2711137):24.34510691,((t88:43.61453378,t89:43.61004904):41.81020001,((t90:38.02855111,t91:37.63886174):25.6182063,t92:63.21307195):21.44263371):1.001441999):128.294883):180.1642534,((((t93:11.49891061,t94:11.3677606):61.49801816,((t95:11.93107304,(t96:4.626101633,t97:4.615104995):6.647926307):42.69934083,t98:53.83920579):17.92581868):67.56357924,(((t99:24.25271397,t100:24.08429857):28.3772211,(((t101:12.83102102,t102:12.5090022):22.17070991,(t103:11.8367177,t104:11.05793691):22.52271151):0.5620867758,t105:33.76110887):15.17674998):1.054430019,((t106:11.89124138,t107:11.74834602):8.169461656,t108:18.95484656):29.89984851):85.12034301):47.66548495,(t109:13.66845066,t110:13.42775698):167.9487691):205.8565257):232.0299868,(((t111:8.937486367,t112:7.830085059):75.23544991,t113:82.52951569):455.1592802,((((t114:4.817223241,t115:4.299202952):38.79873362,((t116:14.53520568,t117:14.22710676):2.514260427,t118:16.48381994):25.79467207):3.144691536,(t119:24.49914135,t120:24.42420434):20.80350828):49.25106912,(t121:42.89815647,t122:41.3437511):51.36787791):441.3089666):79.83073337):3141.157481)"
tree = Nexus.Trees.Tree(tree_text)

In [39]:
len(tree.get_terminals())

193

In [40]:
# Sampling times from tree
#tree    = Nexus.Trees.Tree(newicktree)
forwards = False
leaves  = tree.get_terminals()
heights = np.zeros(len(leaves))	
for i in range(0,len(leaves)):
    heights[i] = tree.sum_branchlength(node=leaves[i])

# Forwards or backwards
if (forwards == True):
    treetimes = heights
else:
    treetimes = max(heights) - heights

In [42]:
leaves

[7,
 8,
 10,
 11,
 13,
 17,
 18,
 21,
 22,
 23,
 26,
 29,
 30,
 31,
 32,
 35,
 36,
 38,
 39,
 41,
 44,
 45,
 47,
 48,
 54,
 55,
 56,
 58,
 60,
 61,
 63,
 64,
 71,
 73,
 74,
 75,
 77,
 78,
 79,
 81,
 82,
 88,
 90,
 91,
 94,
 95,
 97,
 98,
 103,
 104,
 106,
 107,
 110,
 112,
 113,
 114,
 119,
 120,
 122,
 123,
 125,
 127,
 128,
 131,
 132,
 133,
 135,
 140,
 141,
 143,
 144,
 149,
 153,
 154,
 156,
 158,
 161,
 163,
 164,
 166,
 167,
 169,
 173,
 174,
 175,
 176,
 177,
 181,
 182,
 183,
 185,
 186,
 191,
 192,
 193,
 195,
 196,
 197,
 198,
 203,
 205,
 206,
 207,
 212,
 213,
 214,
 216,
 218,
 219,
 223,
 226,
 227,
 229,
 230,
 232,
 233,
 235,
 236,
 246,
 247,
 249,
 250,
 251,
 252,
 254,
 255,
 258,
 261,
 262,
 263,
 265,
 266,
 269,
 270,
 271,
 273,
 274,
 278,
 279,
 282,
 284,
 285,
 287,
 290,
 291,
 292,
 296,
 298,
 299,
 301,
 302,
 304,
 305,
 310,
 311,
 315,
 316,
 317,
 320,
 321,
 324,
 325,
 326,
 331,
 332,
 335,
 337,
 338,
 339,
 343,
 344,
 348,
 349,
 351,
 352,


In [41]:
treetimes

array([6.95899961e+01, 6.99009214e+01, 6.99299383e+01, 7.01237242e+01,
       7.05985777e+01, 7.07458719e+01, 7.07896019e+01, 7.09502747e+01,
       7.24061109e+01, 7.27428906e+01, 7.27817269e+01, 7.29880607e+01,
       7.36036639e+01, 7.36257918e+01, 7.41329264e+01, 7.47255034e+01,
       7.56292491e+01, 7.59811021e+01, 7.64257737e+01, 7.73174568e+01,
       7.73465740e+01, 7.81403614e+01, 7.82073282e+01, 7.86394884e+01,
       7.88751315e+01, 7.96191544e+01, 7.97035743e+01, 8.12494214e+01,
       8.12601967e+01, 8.13931394e+01, 8.17180738e+01, 8.21777518e+01,
       8.22888165e+01, 8.24988569e+01, 8.28126046e+01, 8.37037645e+01,
       8.44161924e+01, 8.52070863e+01, 8.53554290e+01, 8.58849566e+01,
       8.60862936e+01, 8.62699674e+01, 8.64326701e+01, 8.68678593e+01,
       8.71222224e+01, 8.75377536e+01, 8.76889522e+01, 8.79945862e+01,
       8.82814358e+01, 8.84925831e+01, 8.90611364e+01, 8.91120913e+01,
       9.01912617e+01, 9.01934777e+01, 9.03612832e+01, 9.12626198e+01,
      